# 17 Joint

Reads the safety and the linguistic results against each other. Both are
measured on the same requests, so the question this notebook asks is not whether
each moves with the stated age, which Sections 4.2 and 4.3 already answer, but
whether they move in the same shape and on the same scenarios.

Three analyses:

1. **The two ladders.** Refusal Rate and grade level at each of the eight stated
   ages, on one scale each.
2. **Threshold against gradient.** The ladder split into the movement across
   childhood, the step at the statutory boundary, and the movement above it, so
   the two outcomes can be compared as shares of their own range.
3. **Scenario concordance.** Whether the scenarios on which a model changes its
   safety behaviour are the scenarios on which it changes its reading level.

Nothing here is a declared family. Every result is descriptive, carries no
adjusted value, and is reported as such.

## Setup

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / 'scripts'))

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

import analysis
import language
from analysis import FOCUS, MACRO, NAME, ORDER, STATED, STATED_AGE, by_scenario

pd.set_option('display.width', 200, 'display.max_columns', 40)

AGES = [STATED_AGE[name] for name in STATED]
MINOR, ADULT = [a for a in AGES if a < 18], [a for a in AGES if a >= 18]
TABLES = Path.cwd().parent / 'tables'


# Write a finished table to the frozen trail. publish() is the house route and
# refuses a name config/captions.yml does not describe, so until these five
# names are entered there this writes the CSV directly and nothing else.
def freeze(table, name, tier='supplement'):
    path = TABLES / ('main' if tier == 'main' else 'supplement') / f'{name}.csv'
    table.to_csv(path)
    return table

In [ ]:
safety = analysis.load_corpus()
returned = safety[safety['responded']]

reading = language.load()
reading['label'] = reading['model'].map(NAME)
measurable = reading[reading['response_length'] >= language.FLOOR]
stated = measurable[measurable['signal'].eq('stated')]

print(f"{len(returned):,} returned replies, {len(stated):,} measurable at a stated age")

## 1. The two ladders

Refusal is read within Age Restricted scenarios, which is the stratum the
primary hypotheses are tested in and the only one whose expected answer moves
with age. Grade level is read over every stated-age reply, since it carries no
expectation to move.

In [ ]:
refusal = pd.DataFrame(
    {age: {model: by_scenario(returned[(returned['label'] == model)
                                       & returned['scenario_type'].eq(FOCUS)],
                              'refusal', [name]).mean() * 100
           for model in ORDER}
     for name, age in zip(STATED, AGES)}).reindex(ORDER)

grade = pd.DataFrame(
    {age: stated[stated['age'] == age].groupby('label')['fkgl'].mean()
     for age in AGES}).reindex(ORDER)

ladders = pd.concat({'Refusal Rate (%)': refusal, 'Grade Level': grade},
                    names=['Measure', 'Model'])
ladders.columns = [f'Age {age}' for age in AGES]
freeze(ladders.round(2), 'joint_01_ladders', tier='main')
ladders.round(1)

## 2. Threshold against gradient

The ladder is split into three pieces: the movement across the six minor ages,
the single step from seventeen to eighteen, and the movement from eighteen to
twenty-one. The last column is the step as a share of the whole range, which is
what makes the two outcomes comparable despite their different units.

In [ ]:
def shape(ladder, name):
    rows = {}
    for model in ORDER:
        row = ladder.loc[model]
        childhood = row[7] - row[17]
        step = row[17] - row[18]
        adult = row[18] - row[21]
        span = row[7] - row[21]
        rows[model] = {'Across Childhood (7 to 17)': childhood,
                       'Step at the Boundary (17 to 18)': step,
                       'Above the Boundary (18 to 21)': adult,
                       'Full Range (7 to 21)': span,
                       'Step as Share of Range (%)': step / span * 100}
    out = pd.DataFrame(rows).T.reindex(ORDER)
    out.loc[MACRO] = out.mean()
    # The share is a ratio of two of the columns beside it, so the panel figure
    # is that ratio taken on the macro-averaged step and range. Averaging the
    # six per-model shares instead would give a different number, and it would
    # not be the share the macro row's own step and range imply.
    out.loc[MACRO, 'Step as Share of Range (%)'] = (
        out.loc[MACRO, 'Step at the Boundary (17 to 18)']
        / out.loc[MACRO, 'Full Range (7 to 21)'] * 100)
    out.index.name = 'Model'
    return pd.concat({name: out}, names=['Measure', 'Model'])


shapes = pd.concat([shape(refusal, 'Refusal Rate (pp)'),
                    shape(grade, 'Grade Level')])
freeze(shapes.round(2), 'joint_02_shape', tier='main')
shapes.round(1)

## 3. Scenario concordance

For each model and each scenario, the age-induced change in safety behaviour and
the age-induced change in reading level, both as stated minor ages against stated
adult ages. The question is whether the two rank together: if a model rewrites a
scenario for a child, does it also change what it is willing to do on that
scenario?

Spearman is used rather than Pearson because neither difference is expected to be
linear in the other, and the interval is a scenario bootstrap, resampling the
scenarios the correlation is computed over.

In [ ]:
def scenario_shift(frame, column, is_safety):
    part = frame.groupby(['scenario_id', 'age'])[column].mean().unstack()
    part = part.reindex(columns=MINOR + ADULT).dropna()
    return (part[MINOR].mean(axis=1) - part[ADULT].mean(axis=1)) * (100 if is_safety else 1)


rows = []
rng = np.random.default_rng(7)
for model in ORDER:
    left = returned[returned['label'] == model].assign(age=lambda d: d['condition'].map(STATED_AGE))
    right = stated[stated['label'] == model]
    a = scenario_shift(left.dropna(subset=['age']), 'refusal', True)
    b = scenario_shift(right, 'fkgl', False)
    both = pd.concat({'safety': a, 'reading': b}, axis=1).dropna()
    types = returned.drop_duplicates('scenario_id').set_index('scenario_id')['scenario_type']
    for stratum in ['All'] + list(analysis.STRATA):
        keep = both if stratum == 'All' else both[types.reindex(both.index).eq(stratum)]
        # A stratum where refusal never moves has no ranks to correlate. Benign
        # and Rights are flat at every age by design, so they are reported as
        # not applicable rather than as a coefficient scipy cannot define.
        if len(keep) < 8 or keep['safety'].nunique() < 3 or keep['reading'].nunique() < 3:
            continue
        rho = spearmanr(keep['safety'], keep['reading']).statistic
        draws = [spearmanr(*keep.iloc[rng.integers(0, len(keep), len(keep))].values.T).statistic
                 for _ in range(1000)]
        lo, hi = np.nanpercentile(draws, [2.5, 97.5])
        rows.append({'Model': model, 'Scenario Type': stratum, 'Scenarios': len(keep),
                     'rho': rho, '95% CI Lower': lo, '95% CI Upper': hi})

concordance = pd.DataFrame(rows).set_index(['Scenario Type', 'Model'])
freeze(concordance.round(3), 'joint_03_concordance')
concordance.round(2)

## What this notebook writes

| Table | Tier |
|---|---|
| `joint_01_ladders` | main |
| `joint_02_shape` | main |
| `joint_03_concordance` | supplement |

All three are descriptive. None declares a family, none carries an adjusted
value, and Section 4.4 reports them as description.

To move these onto `publish()`, add the three names to `config/captions.yml`
with a `label`, a `tier` and a `kind: table`, then replace `freeze` with
`publish` in the setup cell.